In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import zipfile

dataset_path = '/content/drive/MyDrive/squirrel-v2-audited-coco.zip'

print("File accessible:", os.path.isfile(dataset_path))

if os.path.isfile(dataset_path):
    size_gb = os.path.getsize(dataset_path) / (1024 ** 3)
    print(f"File size: {size_gb:.2f} GB")
    print("Valid ZIP:", zipfile.is_zipfile(dataset_path))

In [ ]:
# Create a folder for the dataset
!mkdir -p /content/data

# Extract the ZIP file
!unzip -q "/content/drive/MyDrive/squirrel-v2-audited-coco.zip" -d /content/data

# Show the extracted files and folders
!ls -lh /content/data

In [ ]:
import json
from pathlib import Path

dataset_dir = Path("/content/data/squirrel-v2-audited-coco")

for split in ["train", "valid", "test"]:
    annotation_file = dataset_dir / split / "_annotations.coco.json"

    with open(annotation_file, "r") as f:
        data = json.load(f)

    print(f"{split}: {len(data['images'])} images, "
          f"{len(data['annotations'])} bounding boxes")

    print(f"Classes: {len(data['categories'])}")
    print("-------------------------")

In [ ]:
# Go to the main Colab directory
%cd /content

# Download the original YOLOv5 repository
!git clone https://github.com/ultralytics/yolov5.git

# Enter the YOLOv5 folder
%cd /content/yolov5

# Install required Python libraries
!pip install -q -r requirements.txt

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import json

annotation_path = "/content/data/squirrel-v2-audited-coco/train/_annotations.coco.json"

with open(annotation_path, "r") as f:
    coco_data = json.load(f)

categories = sorted(coco_data["categories"], key=lambda x: x["id"])

for i, category in enumerate(categories):
    print(f"YOLO Class {i}: {category['name']} (COCO ID: {category['id']})")

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

source = Path("/content/data/squirrel-v2-audited-coco")
output = Path("/content/squirrel-yolo")

for split in ["train", "valid", "test"]:

    # Load COCO annotations
    with open(source / split / "_annotations.coco.json") as f:
        coco = json.load(f)

    # Map COCO class IDs to YOLO class IDs
    categories = sorted(coco["categories"], key=lambda c: c["id"])
    class_map = {c["id"]: i for i, c in enumerate(categories)}

    # Prepare image and label folders
    image_dir = output / "images" / split
    label_dir = output / "labels" / split

    image_dir.parent.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    images = {img["id"]: img for img in coco["images"]}
    labels = defaultdict(list)

    for ann in coco["annotations"]:
        img = images[ann["image_id"]]

        W, H = img["width"], img["height"]
        x, y, w, h = ann["bbox"]

        # Check bounding-box validity
        if not (W > 0 and H > 0 and w > 0 and h > 0
                and x >= 0 and y >= 0
                and x + w <= W + 0.001
                and y + h <= H + 0.001):
            raise ValueError(f"Invalid bounding box: {ann['id']}")

        class_id = class_map[ann["category_id"]]

        # Convert to normalized YOLO coordinates
        xc = (x + w / 2) / W
        yc = (y + h / 2) / H
        nw = w / W
        nh = h / H

        labels[ann["image_id"]].append(
            f"{class_id} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}"
        )

    # Save one label file per image
    for img in coco["images"]:
        filename = Path(img["file_name"])
        if not (source / split / filename).is_file():
            raise FileNotFoundError(filename)

        label_file = label_dir / (filename.stem + ".txt")
        label_file.write_text(
            "\n".join(labels[img["id"]]) +
            ("\n" if labels[img["id"]] else "")
        )

    print(f"{split}: {len(coco['images'])} images converted")

print("COCO to YOLO conversion completed!")

In [ ]:
from pathlib import Path
import os
import shutil

source = Path("/content/data/squirrel-v2-audited-coco")
output = Path("/content/squirrel-yolo")

for split in ["train", "valid", "test"]:
    image_dir = output / "images" / split
    label_dir = output / "labels" / split

    # Remove the folder shortcut, not the original images
    if image_dir.is_symlink():
        image_dir.unlink()

    image_dir.mkdir(parents=True, exist_ok=True)

    # Create direct file links to the original images
    for image in (source / split).iterdir():
        if image.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue

        destination = image_dir / image.name

        if not destination.exists():
            try:
                os.link(image, destination)
            except OSError:
                shutil.copy2(image, destination)

    # Verify image-to-label matching
    images = [
        p for p in image_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    missing = sum(
        not (label_dir / f"{p.stem}.txt").is_file()
        for p in images
    )

    print(f"{split}: {len(images)} images, {missing} missing labels")

print("Folder structure fixed!")

In [ ]:
import json
import yaml
from pathlib import Path

# Read the class names from the original annotations
with open("/content/data/squirrel-v2-audited-coco/train/_annotations.coco.json") as f:
    coco = json.load(f)

categories = sorted(coco["categories"], key=lambda c: c["id"])
class_names = [c["name"] for c in categories]

# Create YOLOv5 dataset configuration
config = {
    "path": "/content/squirrel-yolo",
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "nc": len(class_names),
    "names": class_names
}

# Save data.yaml
yaml_path = "/content/squirrel-yolo/data.yaml"

with open(yaml_path, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("data.yaml created successfully!")
print("Number of classes:", len(class_names))
print("Location:", yaml_path)

In [ ]:
from pathlib import Path

base = Path("/content/squirrel-yolo")

for split in ["train", "valid", "test"]:
    image_dir = base / "images" / split
    label_dir = base / "labels" / split

    images = [
        p for p in image_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    missing = [
        p.name for p in images
        if not (label_dir / (p.stem + ".txt")).is_file()
    ]

    print(f"{split}: {len(images)} images")
    print(f"Missing label files: {len(missing)}")
    print("-------------------------")

In [ ]:
from pathlib import Path

base = Path("/content/squirrel-yolo/labels")
total_boxes = 0
invalid_boxes = 0

for split in ["train", "valid", "test"]:
    for label_file in (base / split).glob("*.txt"):
        for line in label_file.read_text().splitlines():
            values = list(map(float, line.split()))

            if len(values) != 5:
                invalid_boxes += 1
                continue

            cls, x, y, w, h = values

            if not (
                cls.is_integer() and 0 <= cls < 11
                and 0 < w <= 1 and 0 < h <= 1
                and w / 2 <= x <= 1 - w / 2
                and h / 2 <= y <= 1 - h / 2
            ):
                invalid_boxes += 1

            total_boxes += 1

print("Total bounding boxes:", total_boxes)
print("Invalid bounding boxes:", invalid_boxes)

In [ ]:
from pathlib import Path

label_dir = Path("/content/squirrel-yolo/labels")

tolerance = 1e-6
invalid_count = 0
examples = []

for file in label_dir.rglob("*.txt"):
    for line in file.read_text().splitlines():
        values = list(map(float, line.split()))

        if len(values) != 5:
            invalid_count += 1
            continue

        cls, x, y, w, h = values

        valid = (
            cls.is_integer() and 0 <= cls < 11
            and w > 0 and h > 0
            and w <= 1 + tolerance
            and h <= 1 + tolerance
            and x - w/2 >= -tolerance
            and x + w/2 <= 1 + tolerance
            and y - h/2 >= -tolerance
            and y + h/2 <= 1 + tolerance
        )

        if not valid:
            invalid_count += 1
            if len(examples) < 5:
                examples.append((file.name, values))

print("Invalid boxes after rounding tolerance:", invalid_count)
print("Examples:", examples)

In [ ]:
%cd /content/yolov5

from utils.downloads import attempt_download

weights_path = attempt_download("yolov5m.pt")

print("Pretrained model downloaded:", weights_path)

In [ ]:
%cd /content/yolov5

!WANDB_MODE=disabled python train.py \
    --img 640 \
    --batch-size 8 \
    --epochs 10 \
    --data /content/squirrel-yolo/data.yaml \
    --weights yolov5m.pt \
    --device 0 \
    --workers 2 \
    --project /content/drive/MyDrive/YOLOv5_Squirrel_Training \
    --name squirrel_10epochs_fresh \
    --save-period 1

In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/YOLOv5_Squirrel_Training/squirrel_10epochs_fresh/results.csv"

df = pd.read_csv(path)
df.columns = df.columns.str.strip()

columns = [
    "epoch",
    "metrics/precision",
    "metrics/recall",
    "metrics/mAP_0.5",
    "metrics/mAP_0.5:0.95"
]

print(df[columns].tail(1).to_string(index=False))